# AgriTrust — Price Forecaster Training
Trains an ARIMA model (+ optional LSTM) for crop price prediction.

**Output:** `ml_weights/arima_model.pkl`, `ml_weights/lstm_model.h5` (if TensorFlow available)

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'data/training'))
print('Repo root:', REPO_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import MinMaxScaler
    TF = True
    print('TensorFlow available — will train LSTM too')
except ImportError:
    TF = False
    print('TensorFlow not installed — ARIMA only')

## 1. Load Price History

In [ ]:
def load_data():
    try:
        from backend.app.db.session import SessionLocal
        from backend.app.models.listing import Listing
        db = SessionLocal()
        listings = db.query(Listing).filter(Listing.price_per_unit.isnot(None)).order_by(Listing.created_at).all()
        if len(listings) < 30:
            raise ValueError('Not enough listings in DB yet')
        df = pd.DataFrame([{
            'date': l.created_at,
            'price': float(l.price_per_unit),
            'crop': l.product_type
        } for l in listings])
        db.close()
        print(f'Loaded {len(df)} price records from database')
        return df
    except Exception as e:
        print(f'DB unavailable ({e}) — using synthetic data')
        from utils.data_loader import generate_sample_data
        return generate_sample_data('price_history')

df = load_data()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.head()

## 2. Visualise Price History

In [ ]:
for crop, grp in df.groupby('crop'):
    plt.plot(grp['date'], grp['price'], label=crop)
plt.title('Crop Price History')
plt.xlabel('Date')
plt.ylabel('Price (USD/unit)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Train ARIMA (per crop)

In [ ]:
arima_models = {}
crops = df['crop'].unique()

for crop in crops:
    series = df[df['crop'] == crop]['price'].values
    if len(series) < 10:
        print(f'Skipping {crop} — not enough data ({len(series)} points)')
        continue
    split = int(len(series) * 0.8)
    train, test = series[:split], series[split:]
    try:
        m = ARIMA(train, order=(5, 1, 0)).fit()
        forecast = m.forecast(steps=len(test))
        mae = mean_absolute_error(test, forecast)
        print(f'{crop}: MAE=${mae:.4f}  AIC={m.aic:.0f}')
        arima_models[crop] = m
    except Exception as e:
        print(f'{crop}: ARIMA failed — {e}')

print(f'\nTrained ARIMA for: {list(arima_models.keys())}')

## 4. (Optional) Train LSTM on residuals

In [ ]:
lstm_model = None
lstm_scaler = None

if TF:
    # Use the first crop with enough data
    target_crop = list(arima_models.keys())[0]
    series = df[df['crop'] == target_crop]['price'].values
    split = int(len(series) * 0.8)
    train = series[:split]
    arima_pred = arima_models[target_crop].predict(start=0, end=len(train)-1)
    residuals = train - arima_pred

    lstm_scaler = MinMaxScaler()
    res_scaled = lstm_scaler.fit_transform(residuals.reshape(-1, 1)).flatten()

    SEQ = 30
    X, y = [], []
    for i in range(len(res_scaled) - SEQ):
        X.append(res_scaled[i:i+SEQ])
        y.append(res_scaled[i+SEQ])
    X, y = np.array(X), np.array(y)

    if len(X) > 50:
        sp = int(0.8 * len(X))
        X_tr, X_te = X[:sp].reshape(-1, SEQ, 1), X[sp:].reshape(-1, SEQ, 1)
        y_tr, y_te = y[:sp], y[sp:]

        lstm_model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(SEQ, 1)),
            Dropout(0.2),
            LSTM(32),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        lstm_model.compile(optimizer='adam', loss='mse')
        lstm_model.fit(X_tr, y_tr, epochs=50, batch_size=32,
                       validation_data=(X_te, y_te),
                       callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
                       verbose=1)
        print('LSTM training complete')
    else:
        print('Not enough residual data for LSTM')
else:
    print('Skipping LSTM (TensorFlow not installed)')

## 5. 30-Day Forecast Preview

In [ ]:
for crop, m in arima_models.items():
    forecast = m.forecast(steps=30)
    plt.plot(forecast, label=crop)

plt.title('30-Day Price Forecast')
plt.xlabel('Days ahead')
plt.ylabel('Price (USD/unit)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Save

In [ ]:
WEIGHTS = os.path.join(REPO_ROOT, 'ml_weights')
os.makedirs(WEIGHTS, exist_ok=True)
joblib.dump(arima_models, f'{WEIGHTS}/arima_model.pkl')
print('Saved arima_model.pkl')

if lstm_model and lstm_scaler:
    lstm_model.save(f'{WEIGHTS}/lstm_model.h5')
    joblib.dump(lstm_scaler, f'{WEIGHTS}/price_scaler.pkl')
    print('Saved lstm_model.h5 + price_scaler.pkl')